### Getting Weather data
This notebook uses governnment website API to collect and save weather data

In [50]:
import requests
import pandas as pd
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [38]:
def fetch_weather_data(session, endpoint, date):
    url = f"https://api.data.gov.sg/v1/environment/{endpoint}?date={date}"
    response = session.get(url)
    return response.json()

In [39]:
def transform_items(items, station_id):
    df = flatten_json(input_data=items, token='readings')

    if df.empty:
        return None

    expanded = pd.DataFrame({
        'name': df.timestamp.repeat(df.readings.str.len()),
        'subGroup': df.readings.sum()
    })

    expanded = pd.concat(
        [expanded.drop(['subGroup'], axis=1),
         expanded['subGroup'].apply(pd.Series)],
        axis=1
    )

    return expanded[expanded['station_id'] == station_id]

In [40]:
import requests
import re
import json
def flatten_json(input_data, sep='_', token='parameters', new_token=None):
    '''
    Flattern nested json, assume only 1 level nested with token specified

    Input:
    input_data: data returned from mongodb query
    sep: separator for column name
    token: if 'token' in the column name, it will be removed from the column name if 'new_token' is None,
           if 'new_token' is not None, 'token' will be replaced by 'new_token'.
           'token' should be at the beginning of the column name
    new_token: new token name to replace token in column name

    Return:
    df: flattened dataframe
    '''
    pattern = re.compile('^'+token+sep)
    df = pd.json_normalize(list(input_data), sep=sep)

    # rename the columns
    col_map = {}
    for k in df.columns.values:
        if pattern.match(k):
            if new_token is None:
                col_map[k] = sep.join(k.split(sep)[1:])
            else:
                col_map[k] = sep.join([new_token]+k.split(sep)[1:])
    df.rename(columns=col_map, inplace=True)

    return df

In [41]:
def get_data_gov(data_type, station_id, date_start, date_end, missing):
    session = requests.Session()
    final = pd.DataFrame()

    endpoints = {
        "humidity": "relative-humidity",
        "air_temp": "air-temperature",
        "wind_speed": "wind-speed",
        "wind_direction": "wind-direction",
        "rainfall": "rainfall"
    }

    # metadata shortcut
    if data_type == "metadata":
        url = "https://api.data.gov.sg/v1/environment/"+endpoints[data_type]+"?date={}"
        r = session.get(url.format(date_end))
        return pd.json_normalize(r.json()['metadata']['stations'])

    endpoint = endpoints.get(data_type)
    if not endpoint:
        raise ValueError(f"Unknown data type: {data_type}")

    for date in pd.date_range(date_start, date_end, freq='D'):
        d = str(date.date())

        json_data = fetch_weather_data(session, endpoint, d)
        items = json_data.get("items", [])
        logger.info(f"Processing date {d}")

        transformed = transform_items(items, station_id)

        if transformed is None or transformed.empty:
            logger.warning(f"Missing data for {d}")
            missing.append(d)
        else:
            final = pd.concat([final, transformed], axis=0)


    return final, missing

In [42]:
def format_weather_data(df, df_name):
    date_list = df['name'].tolist()
    d =[]
    for date in date_list:
        day = datetime.fromisoformat(date[:-6])
        day = day.strftime('%Y-%m-%d %H:%M:%S')
        day= pd.to_datetime(day, errors='coerce')
        d.append(day)

    df['datetime'] = d
    df = df.set_index('datetime')
    df = df['value'].squeeze()
    df = df.rename(df_name)
    return df

In [43]:
missing=[]
wind_direction, missing = get_data_gov('wind_direction','S24','2019-01-01','2019-01-02',missing_3)
wind_direction, missing

INFO:__main__:Processing date 2019-01-01
INFO:__main__:Processing date 2019-01-02


(                           name station_id  value
 0     2019-01-01T00:01:00+08:00        S24    354
 1     2019-01-01T00:02:00+08:00        S24    352
 2     2019-01-01T00:03:00+08:00        S24    348
 3     2019-01-01T00:04:00+08:00        S24    345
 4     2019-01-01T00:05:00+08:00        S24    343
 ...                         ...        ...    ...
 1433  2019-01-02T23:55:00+08:00        S24    302
 1434  2019-01-02T23:56:00+08:00        S24    305
 1435  2019-01-02T23:57:00+08:00        S24    303
 1436  2019-01-02T23:58:00+08:00        S24    305
 1437  2019-01-02T23:59:00+08:00        S24    302
 
 [2737 rows x 3 columns],
 [])

In [ ]:
wind_direction_list.to_csv('CMS - wind_dir - jan2019.csv', index = True)

In [44]:
missing=[]
humidity, missing = get_data_gov('humidity','S24','2019-01-01','2019-01-02',missing)
humidity, missing

INFO:__main__:Processing date 2019-01-01
INFO:__main__:Processing date 2019-01-02


(                           name station_id  value
 0     2019-01-01T00:01:00+08:00        S24   90.7
 1     2019-01-01T00:02:00+08:00        S24   90.7
 2     2019-01-01T00:03:00+08:00        S24   90.6
 3     2019-01-01T00:04:00+08:00        S24   90.5
 4     2019-01-01T00:05:00+08:00        S24   90.6
 ...                         ...        ...    ...
 1433  2019-01-02T23:55:00+08:00        S24   77.7
 1434  2019-01-02T23:56:00+08:00        S24   77.9
 1435  2019-01-02T23:57:00+08:00        S24   77.9
 1436  2019-01-02T23:58:00+08:00        S24   77.9
 1437  2019-01-02T23:59:00+08:00        S24   78.0
 
 [2753 rows x 3 columns],
 [])

In [45]:
missing=[]
air_temp, missing = get_data_gov('air_temp','S24','2019-01-01','2019-01-02',missing)
air_temp, missing

INFO:__main__:Processing date 2019-01-01
INFO:__main__:Processing date 2019-01-02


(                           name station_id  value
 0     2019-01-01T00:01:00+08:00        S24   25.8
 1     2019-01-01T00:02:00+08:00        S24   25.8
 2     2019-01-01T00:03:00+08:00        S24   25.7
 3     2019-01-01T00:04:00+08:00        S24   25.7
 4     2019-01-01T00:05:00+08:00        S24   25.7
 ...                         ...        ...    ...
 1433  2019-01-02T23:55:00+08:00        S24   28.5
 1434  2019-01-02T23:56:00+08:00        S24   28.5
 1435  2019-01-02T23:57:00+08:00        S24   28.5
 1436  2019-01-02T23:58:00+08:00        S24   28.5
 1437  2019-01-02T23:59:00+08:00        S24   28.5
 
 [2751 rows x 3 columns],
 [])

In [46]:
missing=[]
wind_speed, missing = get_data_gov('wind_speed','S24','2019-01-01','2019-01-02',missing)
wind_speed, missing

INFO:__main__:Processing date 2019-01-01
INFO:__main__:Processing date 2019-01-02


(                           name station_id  value
 0     2019-01-01T00:01:00+08:00        S24    2.6
 1     2019-01-01T00:02:00+08:00        S24    2.5
 2     2019-01-01T00:03:00+08:00        S24    2.6
 3     2019-01-01T00:04:00+08:00        S24    2.7
 4     2019-01-01T00:05:00+08:00        S24    2.8
 ...                         ...        ...    ...
 1433  2019-01-02T23:55:00+08:00        S24    4.5
 1434  2019-01-02T23:56:00+08:00        S24    4.8
 1435  2019-01-02T23:57:00+08:00        S24    5.0
 1436  2019-01-02T23:58:00+08:00        S24    5.0
 1437  2019-01-02T23:59:00+08:00        S24    5.1
 
 [2754 rows x 3 columns],
 [])

In [47]:
missing=[]
rainfall, missing = get_data_gov('rainfall','S24','2019-01-01','2019-01-02',missing)
rainfall, missing

INFO:__main__:Processing date 2019-01-01
INFO:__main__:Processing date 2019-01-02


(                          name station_id  value
 0    2019-01-01T00:05:00+08:00        S24    0.0
 1    2019-01-01T00:10:00+08:00        S24    0.0
 2    2019-01-01T00:15:00+08:00        S24    0.0
 3    2019-01-01T00:20:00+08:00        S24    0.0
 4    2019-01-01T00:25:00+08:00        S24    0.0
 ..                         ...        ...    ...
 282  2019-01-02T23:35:00+08:00        S24    0.0
 283  2019-01-02T23:40:00+08:00        S24    0.0
 284  2019-01-02T23:45:00+08:00        S24    0.0
 285  2019-01-02T23:50:00+08:00        S24    0.0
 286  2019-01-02T23:55:00+08:00        S24    0.0
 
 [574 rows x 3 columns],
 [])